# Baseline Evaluation — Llama 3.2-1B on 1,000 Test Examples

**Project:** LoRA Fine-Tuning for Text Summarization  
**Model:** Llama 3.2-1B (no fine-tuning)  
**Dataset:** CNN/DailyMail (version 3.0.0)  
**Purpose:** Establish a baseline by evaluating the raw, untrained Llama 3.2-1B model on 1,000 test examples. This gives us a reference point to measure how much LoRA fine-tuning actually improves performance.

---

### What is a Baseline?
A baseline is the performance of a model **before any fine-tuning**. We run the pre-trained Llama model as-is on summarization tasks. Since it has never been specifically trained for summarization, we expect low scores but this is intentional. It shows us where we started and how much LoRA improved things.

---
### Runtime Estimate
- **GPU:** A100 (Colab Pro)
- **Time:** ~30 minutes for 1,000 examples (sequential evaluation)

## Step 1 — Install Required Libraries
We install all the necessary Python libraries:
- `transformers` — HuggingFace library to load and run pre-trained models like Llama
- `datasets` — HuggingFace library to load benchmark datasets like CNN/DailyMail
- `evaluate` — HuggingFace library for computing BLEU and ROUGE scores
- `rouge-score` — backend used by the evaluate library for ROUGE computation
- `nltk` — used internally for tokenization during BLEU calculation
- `accelerate` — helps run models efficiently on GPU

In [1]:
!pip install transformers datasets evaluate rouge-score nltk accelerate -q
print("All libraries installed successfully!")

All libraries installed successfully!


## Step 2 — Check GPU Availability
We verify that a GPU is available and accessible via CUDA.

**What is CUDA?**  
CUDA is NVIDIA's platform that allows Python code to run on the GPU instead of CPU. Running a large language model like Llama on CPU would take days — on a GPU it takes minutes. PyTorch uses CUDA automatically when a GPU is available.

In [2]:
import torch

# Check if GPU is available
if torch.cuda.is_available():
    print(f"   GPU is available!")
    print(f"   GPU Name     : {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   No GPU found. Please enable GPU in Colab: Runtime > Change runtime type > A100 GPU")

   GPU is available!
   GPU Name     : NVIDIA A100-SXM4-40GB
   GPU Memory   : 42.4 GB


## Step 3 — Authenticate with HuggingFace
Llama 3.2 is a gated model on HuggingFace — meaning you need to:
1. Have a HuggingFace account
2. Accept Meta's license at: https://huggingface.co/meta-llama/Llama-3.2-1B
3. Generate an access token at: https://huggingface.co/settings/tokens

Paste your token when prompted below.

In [3]:
from huggingface_hub import login

# This will prompt you to enter your HuggingFace access token
# Get your token from: https://huggingface.co/settings/tokens
login()
print(" Logged in to HuggingFace!")

 Logged in to HuggingFace!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Step 4 — Load the CNN/DailyMail Test Dataset (1,000 examples)
We load only 1,000 examples from the CNN/DailyMail test split for this notebook.

**Why CNN/DailyMail?**  
It is the industry-standard benchmark dataset for news summarization research. It contains ~312k article-summary pairs from CNN and Daily Mail news articles. Using a public, well-known dataset makes our results reproducible and comparable with other research.

**Why only 1,000 examples here?**  
This is a quick proof-of-concept evaluation. Running on 1,000 examples takes ~30 minutes on a A100 GPU. For the full 11,490-example baseline, see `baseline_eval_11k.ipynb`.

In [4]:
from datasets import load_dataset

print("Loading CNN/DailyMail test set (1,000 examples)...")

# Load the test split and select first 1,000 examples
# Version 3.0.0 is the standard version used in research
test_dataset = load_dataset("cnn_dailymail", "3.0.0", split="test")
test_dataset_1k = test_dataset.select(range(1000))

print(f"   Loaded {len(test_dataset_1k)} test examples")
print(f"\nDataset columns: {test_dataset_1k.column_names}")
print(f"\n--- Sample Example ---")
print(f"Article (first 300 chars) : {test_dataset_1k[0]['article'][:300]}")
print(f"\nReference Summary         : {test_dataset_1k[0]['highlights']}")

Loading CNN/DailyMail test set (1,000 examples)...
   Loaded 1000 test examples

Dataset columns: ['article', 'highlights', 'id']

--- Sample Example ---
Article (first 300 chars) : (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the cou

Reference Summary         : Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


## Step 5 — Load the Llama 3.2-1B Model and Tokenizer

**What is a Tokenizer?**  
A tokenizer converts raw text into numbers (tokens) that the model understands. For example, `"Hello world"` becomes `[15043, 1917]`. It also does the reverse — converting model output numbers back into readable text.

**What is AutoModelForCausalLM?**  
This loads a causal language model — a model that generates text by predicting the next token based on all previous tokens. Llama is a causal LM. "Auto" means HuggingFace automatically picks the right model architecture based on the model name.

**Why `torch_dtype=torch.float16`?**  
This loads the model in 16-bit floating point instead of 32-bit. It cuts memory usage roughly in half, allowing us to run a 1B parameter model on a A100 GPU (40GB VRAM).

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "meta-llama/Llama-3.2-1B"

print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Llama does not have a dedicated padding token, so we use the end-of-sequence token instead
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model {MODEL_NAME}...")
print("This may take a few minutes on first run (downloading ~2.5GB)...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # Use 16-bit precision to reduce memory usage
    device_map="auto"            # Automatically place model on GPU if available
)

# Set model to evaluation mode — disables dropout layers used only during training
model.eval()

print(f"\n   Model loaded successfully!")
print(f"   Model device : {next(model.parameters()).device}")
print(f"   Parameters   : {sum(p.numel() for p in model.parameters()):,}")

Loading tokenizer for meta-llama/Llama-3.2-1B...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model meta-llama/Llama-3.2-1B...
This may take a few minutes on first run (downloading ~2.5GB)...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]


   Model loaded successfully!
   Model device : cuda:0
   Parameters   : 1,235,814,400


## Step 6 — Test on a Single Example
Before running the full evaluation, we test on one example to make sure the model loads correctly and generates output.

**How does text generation work here?**
1. We build a prompt: `"Summarize the following article:\n\n{article}\n\nSummary:"`
2. The tokenizer converts this prompt into token IDs
3. The model generates new tokens after `"Summary:"`
4. The tokenizer decodes the output tokens back into text
5. We extract only the part after `"Summary:"` as the generated summary

**What is `torch.no_grad()`?**  
During inference (not training), we don't need to compute gradients. `torch.no_grad()` disables gradient tracking, saving memory and speeding up inference significantly.

In [6]:
import time

# Pick the first test example
test_example = test_dataset_1k[0]

# Build the prompt — this tells the model what task to perform
# The model will generate text that continues after "Summary:"
prompt = f"Summarize the following article:\n\n{test_example['article']}\n\nSummary:"

# Tokenize the prompt
# truncation=True   : if article is too long, cut it to max_length
# max_length=512    : maximum number of input tokens
# return_tensors='pt': return PyTorch tensors (not plain lists)
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# Move inputs to the same device as the model (GPU)
# Model and data must be on the same device or PyTorch will throw an error
inputs = {k: v.to(model.device) for k, v in inputs.items()}

start_time = time.time()

# Generate the summary
# torch.no_grad() disables gradient computation — we are only doing inference, not training
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,                        # Generate at most 100 new tokens
        do_sample=False,                           # Greedy decoding — always pick the most probable token
        pad_token_id=tokenizer.eos_token_id        # Use EOS token for padding (Llama has no dedicated pad token)
    )

elapsed = time.time() - start_time

# Decode the output token IDs back into readable text
# skip_special_tokens=True removes tokens like <eos>, <pad> from the output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract only the part after "Summary:" — the full output includes the input prompt
generated_summary = generated_text.split("Summary:")[-1].strip()[:200]

print("=" * 60)
print("SINGLE EXAMPLE TEST")
print("=" * 60)
print(f"Article (first 200 chars) : {test_example['article'][:200]}")
print(f"\nReference Summary         : {test_example['highlights']}")
print(f"\nGenerated Summary         : {generated_summary}")
print(f"\nGeneration Time           : {elapsed:.2f} seconds")
print("\n   Single example test passed! Proceeding to full evaluation...")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SINGLE EXAMPLE TEST
Article (first 200 chars) : (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territor

Reference Summary         : Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

Generated Summary         : Summarize the following article:

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alle

Generation Time           : 2.59 seconds

   Single example test passed! Proceeding to full evaluation...


## Step 7 — Run Baseline Evaluation on 1,000 Examples
We now run the untrained Llama model on all 1,000 test examples and collect generated summaries.

**Why sequential evaluation here?**  
For 1,000 examples on a A100 GPU, sequential (one-by-one) evaluation is manageable (~30 mins). For larger evaluations (11k examples), we use batched evaluation which is 13x faster — see `baseline_eval_11k.ipynb`.

**What are we collecting?**
- `all_summaries` — the model's generated summaries
- `all_references` — the ground truth summaries from the dataset

We compare these two lists using BLEU and ROUGE metrics in the next step.

In [7]:
from tqdm import tqdm
import time

all_summaries = []    # Will store model-generated summaries
all_references = []   # Will store ground truth reference summaries

print("Starting baseline evaluation on 1,000 test examples...")
print("Estimated time: ~30 minutes on A100 GPU")
print("=" * 60)

start_total = time.time()

# Loop through all 1,000 test examples
for i, example in enumerate(tqdm(test_dataset_1k, desc="Evaluating")):

    # Build the same prompt format used consistently across all experiments
    prompt = f"Summarize the following article:\n\n{example['article']}\n\nSummary:"

    # Tokenize and move to GPU
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate summary — no gradients needed during evaluation
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode and extract the summary portion
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_summary = generated_text.split("Summary:")[-1].strip()

    all_summaries.append(generated_summary)
    all_references.append(example["highlights"])

    # Print a progress update every 100 examples
    if (i + 1) % 100 == 0:
        elapsed = time.time() - start_total
        print(f"   [{i+1}/1000] completed | Time elapsed: {elapsed/60:.1f} mins")

total_time = time.time() - start_total
print(f"\n   Evaluation complete!")
print(f"   Total examples evaluated : {len(all_summaries)}")
print(f"   Total time               : {total_time/60:.1f} minutes")

Starting baseline evaluation on 1,000 test examples...
Estimated time: ~30 minutes on A100 GPU


Evaluating:  10%|█         | 100/1000 [02:41<26:55,  1.79s/it]

   [100/1000] completed | Time elapsed: 2.7 mins


Evaluating:  20%|██        | 200/1000 [05:39<25:37,  1.92s/it]

   [200/1000] completed | Time elapsed: 5.7 mins


...

   [800/1000] completed | Time elapsed: 23.2 mins


Evaluating:  90%|█████████ | 900/1000 [26:15<02:56,  1.76s/it]

   [900/1000] completed | Time elapsed: 26.3 mins


Evaluating: 100%|██████████| 1000/1000 [29:19<00:00,  1.76s/it]

   [1000/1000] completed | Time elapsed: 29.3 mins

   Evaluation complete!
   Total examples evaluated : 1000
   Total time               : 29.3 minutes


## Step 8 — Calculate BLEU and ROUGE Scores

We now compare the model's generated summaries against the reference summaries using four metrics:

| Metric | What it measures |
|--------|------------------|
| **BLEU-4** | Overlap of 4-word phrases between generated and reference summary |
| **ROUGE-1** | Overlap of individual words (unigrams) |
| **ROUGE-2** | Overlap of 2-word phrases (bigrams) |
| **ROUGE-L** | Longest common subsequence between generated and reference |

**Note:** For abstractive summarization, scores in the range 10–40 are typical. The model paraphrases rather than copying — so low scores don't mean failure, they are expected. What matters is the **improvement** after fine-tuning.

In [10]:
from evaluate import load
import numpy as np

print("Calculating BLEU and ROUGE scores...")
print("=" * 60)

# Load evaluation metrics from HuggingFace evaluate library
bleu_metric = load("bleu")
rouge_metric = load("rouge")

# Both BLEU and ROUGE receive raw strings — the evaluate library handles tokenization internally
# predictions : list of generated summary strings
# references  : for BLEU, each reference is wrapped in a list (supports multiple references per example)
#               for ROUGE, plain list of strings is fine
bleu_result = bleu_metric.compute(
    predictions=all_summaries,
    references=[[r] for r in all_references]   # each reference wrapped in a list
)

rouge_result = rouge_metric.compute(
    predictions=all_summaries,
    references=all_references                   # plain list of strings
)

# Extract BLEU-4 score (index 3 = 4-gram precision) and scale to 0-100
bleu4  = bleu_result['precisions'][3] * 100
rouge1 = rouge_result['rouge1'] * 100
rouge2 = rouge_result['rouge2'] * 100
rougeL = rouge_result['rougeL'] * 100

print(f"\n{'='*60}")
print(f"  BASELINE RESULTS — Llama 3.2-1B | 1,000 Test Examples")
print(f"{'='*60}")
print(f"  BLEU-4   : {bleu4:.2f}")
print(f"  ROUGE-1  : {rouge1:.2f}")
print(f"  ROUGE-2  : {rouge2:.2f}")
print(f"  ROUGE-L  : {rougeL:.2f}")
print(f"{'='*60}")
print(f"\nNote: These are baseline scores with NO fine-tuning.")
print(f"Compare with LoRA fine-tuned results in the lora_1k notebooks.")

Calculating BLEU and ROUGE scores...

  BASELINE RESULTS — Llama 3.2-1B | 1,000 Test Examples
  BLEU-4   : 0.95
  ROUGE-1  : 15.68
  ROUGE-2  : 6.70
  ROUGE-L  : 11.32

Note: These are baseline scores with NO fine-tuning.
Compare with LoRA fine-tuned results in the lora_1k notebooks.


## Step 9 — Save Results
We save the scores to a JSON file for easy reference and comparison across notebooks.

In [11]:
import json

# Store all results in a dictionary
baseline_results_1k = {
    "experiment"  : "Baseline — Llama 3.2-1B (no fine-tuning)",
    "model"       : "meta-llama/Llama-3.2-1B",
    "test_samples": 1000,
    "bleu_4"      : round(bleu4, 4),
    "rouge_1"     : round(rouge1, 4),
    "rouge_2"     : round(rouge2, 4),
    "rouge_L"     : round(rougeL, 4)
}

# Save to JSON file
with open("baseline_results_1k.json", "w") as f:
    json.dump(baseline_results_1k, f, indent=2)

print("   Results saved to baseline_results_1k.json")
print(json.dumps(baseline_results_1k, indent=2))

   Results saved to baseline_results_1k.json
{
  "experiment": "Baseline \u2014 Llama 3.2-1B (no fine-tuning)",
  "model": "meta-llama/Llama-3.2-1B",
  "test_samples": 1000,
  "bleu_4": 0.9546,
  "rouge_1": 15.6771,
  "rouge_2": 6.7044,
  "rouge_L": 11.3215
}
